In [12]:
import numpy as np
import random

# Gerar Matriz de Latência Simétrica 10x10 (D)
np.random.seed(42)
D = np.random.randint(10, 100, size=(10, 10))
D = (D + D.T) / 2
np.fill_diagonal(D, 0)

# Estrutura Union-Find para deteção de ciclos (garante validade da árvore)
class UnionFind:
    def __init__(self, n):
        self.pai = list(range(n))
    def find(self, i):
        if self.pai[i] == i: return i
        self.pai[i] = self.find(self.pai[i])
        return self.pai[i]
    def union(self, i, j):
        raiz_i, raiz_j = self.find(i), self.find(j)
        if raiz_i != raiz_j:
            self.pai[raiz_i] = raiz_j
            return True
        return False

def gerar_arvore_aleatoria():
    uf = UnionFind(10)
    arestas = [(i, j) for i in range(10) for j in range(i+1, 10)]
    random.shuffle(arestas)
    arvore = []
    latencia = 0
    for u, v in arestas:
        if uf.union(u, v):
            arvore.append((u, v))
            latencia += D[u][v]
            if len(arvore) == 9: break
    return arvore, latencia

def executar_aco(num_formigas=20, iteracoes=50, rho=0.2, alpha=1.0, beta=2.0):
    tau = np.ones((10, 10)) # Matriz de Feromonas inicial
    eta = 1.0 / (D + 1e-10) # Heurística (inverso da latência)

    melhor_arvore_global = None
    menor_latencia_global = float('inf')

    for _ in range(iteracoes):
        melhor_arvore_iteracao = None
        menor_latencia_iteracao = float('inf')

        for _ in range(num_formigas):
            uf = UnionFind(10)
            arvore_atual = []
            latencia_atual = 0
            arestas_disponiveis = [(i, j) for i in range(10) for j in range(i+1, 10)]

            while len(arvore_atual) < 9:
                probabilidades = []
                arestas_validas = []

                # Avalia apenas arestas que não formam ciclos
                for u, v in arestas_disponiveis:
                    if uf.find(u) != uf.find(v):
                        p = (tau[u][v] ** alpha) * (eta[u][v] ** beta)
                        probabilidades.append(p)
                        arestas_validas.append((u, v))

                # Escolha roleta
                probabilidades = np.array(probabilidades) / sum(probabilidades)
                idx_escolhido = np.random.choice(len(arestas_validas), p=probabilidades)
                u_esc, v_esc = arestas_validas[idx_escolhido]

                uf.union(u_esc, v_esc)
                arvore_atual.append((u_esc, v_esc))
                latencia_atual += D[u_esc][v_esc]
                arestas_disponiveis.remove((u_esc, v_esc))

            if latencia_atual < menor_latencia_iteracao:
                menor_latencia_iteracao = latencia_atual
                melhor_arvore_iteracao = arvore_atual

            if latencia_atual < menor_latencia_global:
                menor_latencia_global = latencia_atual
                melhor_arvore_global = arvore_atual

        # Atualização de Feromonas aplicada apenas à melhor topologia da iteração
        tau *= (1 - rho)
        for u, v in melhor_arvore_iteracao:
            tau[u][v] += 1.0 / menor_latencia_iteracao
            tau[v][u] += 1.0 / menor_latencia_iteracao

    return melhor_arvore_global, menor_latencia_global

# Execução e Relatório
print("LAB 03 - ACO PARA TOPOLOGIA DE REDE\n" + "="*40)
_, latencia_aleatoria = gerar_arvore_aleatoria()
melhor_arvore_aco, latencia_aco = executar_aco()

ganho_percentual = ((latencia_aleatoria - latencia_aco) / latencia_aleatoria) * 100

matriz_adj = np.zeros((10, 10), dtype=int)
for u, v in melhor_arvore_aco:
    matriz_adj[u][v] = 1
    matriz_adj[v][u] = 1

print(f"Latência Topologia Aleatória: {latencia_aleatoria:.2f}")
print(f"Latência Otimizada (ACO): {latencia_aco:.2f}")
print(f"Ganho de Desempenho: {ganho_percentual:.2f}%\n")
print("Matriz de Adjacência Final (10x10):")
print(matriz_adj)

LAB 03 - ACO PARA TOPOLOGIA DE REDE
Latência Topologia Aleatória: 565.00
Latência Otimizada (ACO): 253.00
Ganho de Desempenho: 55.22%

Matriz de Adjacência Final (10x10):
[[0 1 0 0 1 0 0 0 0 0]
 [1 0 1 0 0 0 0 0 1 0]
 [0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0]
 [1 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 1 0]
 [0 0 0 1 0 0 1 0 0 1]
 [0 1 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 1 0 0]]
